## **Model.py**

In [1]:
import torch
import torch.nn as nn
import math


class InputEmbeddings(nn.Module):
  def __init__(self, d_model, vocab_size):
    super().__init__()
    self.d_model = d_model
    self.vocab_size = vocab_size
    self.embedding = nn.Embedding(vocab_size, d_model)
  def forward(self, x):
    return self.embedding(x) * math.sqrt(self.d_model)

class PositionalEncoding(nn.Module):
  def __init__(self, d_model: int, seq_len: int, dropout: float)->None:
    super().__init__()
    self.d_model = d_model
    self.seq_len = seq_len
    self.dropout = nn.Dropout(dropout)

    pe = torch.zeros(seq_len, d_model)

    position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)

    pe = pe.unsqueeze(0)

    self.register_buffer('pe', pe)

  def forward(self, x):
    x = x+ (self.pe[:, :x.shape[1], :]).requires_grad_(False)
    return self.dropout(x)



class LayerNormalization(nn.Module):
  def __init__(self, eps: float=10**-6)->None:
    super().__init__()
    self.eps = eps
    self.alpha = nn.Parameter(torch.ones(1))
    self.beta = nn.Parameter(torch.zeros(1))

  def forward(self, x):
    mean = x.mean(dim = -1, keepdim=True)
    std = x.std(dim = -1, keepdim=True)
    return self.alpha * (x - mean) / (std + self.eps) + self.beta



class FeedForwardBlock(nn.Module):
  def __init__(self, d_model: int, d_ff: int, dropout: float)->None:
    super().__init__()
    self.linear_1 = nn.Linear(d_model, d_ff)
    self.dropout = nn.Dropout(dropout)
    self.linear_2 = nn.Linear(d_ff, d_model)

  def forward(self, x):
    return self.linear_2(self.dropout(torch.relu(self.linear_1(x))))


class MultiHeadAttentionBlock(nn.Module):
  def __init__(self, d_model:int, h: int, dropout: float)->None:
    super().__init__()
    self.d_model = d_model
    self.h = h
    assert d_model%h == 0, "d_model is not divisible by h"

    self.d_k = d_model//h
    self.w_q = nn.Linear(d_model, d_model)
    self.w_k = nn.Linear(d_model, d_model)
    self.w_v = nn.Linear(d_model, d_model)

    self.w_o = nn.Linear(d_model, d_model)
    self.dropout = nn.Dropout(dropout)

  @staticmethod
  def attention(query, key, value,mask, dropout: nn.Dropout):
    d_k = query.shape[-1]

    attention_scores = (query @ key.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
      attention_scores.masked_fill_(mask == 0, -1e9)
    attention_scores = attention_scores.softmax(dim=-1)
    if dropout is not None:
      attention_scores = dropout(attention_scores)

    return (attention_scores @ value), attention_scores

  def forward(self, q, k, v, mask):
    query = self.w_q(q)
    key = self.w_k(k)
    value = self.w_v(v)

    query = query.view(query.shape[0], query.shape[1], self.h, self.d_k).transpose(1, 2)
    key = key.view(key.shape[0], key.shape[1], self.h, self.d_k).transpose(1, 2)
    value = value.view(value.shape[0], value.shape[1], self.h, self.d_k).transpose(1, 2)

    x, self.attention_scores = MultiHeadAttentionBlock.attention(query, key, value, mask, self.dropout)

    x = x.transpose(1, 2).contiguous().view(x.shape[0], -1, self.h*self.d_k)

    return self.w_o(x)

class ResidualConnection(nn.Module):
  def __init__(self, dropout: float)->None:
    super().__init__()
    self.dropout = nn.Dropout(dropout)
    self.norm = LayerNormalization()

  def forward(self, x, sublayer):
    return x + self.dropout(sublayer(self.norm(x)))




class DecoderBlock(nn.Module):
  def __init__(self, self_attention_block: MultiHeadAttentionBlock, feed_forward_block: FeedForwardBlock, dropout: float)->None:
    super().__init__()
    self.self_attention_block = self_attention_block
    self.feed_forward_block = feed_forward_block
    self.residual_connections = nn.ModuleList([ResidualConnection(dropout) for _ in range(2)])


  def forward(self, x, mask):
    x = self.residual_connections[0](x, lambda x: self.self_attention_block(x, x, x, mask))
    x = self.residual_connections[1](x, self.feed_forward_block)
    return x


class Decoder(nn.Module):
  def __init__(self, layers: nn.ModuleList)->None:
    super().__init__()
    self.layers = layers
    self.norm = LayerNormalization()

  def forward(self, x, mask):
    for layer in self.layers:
      x = layer(x, mask)
    return self.norm(x)


class ProjectionLayer(nn.Module):

  def __init__(self, d_model: int, vocab_size: int)->None:
    super().__init__()
    self.proj = nn.Linear(d_model, vocab_size)


  def forward(self, x):
    return torch.log_softmax(self.proj(x), dim = -1)



class Transformer(nn.Module):
  def __init__(self,
               decoder: Decoder,
               embed: InputEmbeddings,
               pos: PositionalEncoding,
               projection_layer: ProjectionLayer
               ):
    super().__init__()
    self.decoder = decoder
    self.embed = embed
    self.pos = pos
    self.projection = projection_layer


  def decode(self, src, mask):
    src = self.embed(src)
    src = self.pos(src)
    return self.decoder(src, mask)

  def project(self, x):
    return self.projection(x)


def build_transformer(vocab_size: int,
                      seq_len: int,
                      d_model: int = 512,
                      N: int = 12,
                      h: int = 8,
                      dropout: float = 0.1,
                      d_ff: int = 2048)-> Transformer:

  embed = InputEmbeddings(d_model, vocab_size)

  pos = PositionalEncoding(d_model, seq_len, dropout)

  encoder_blocks = []

  decoder_blocks = []
  for _ in range(N):
    decoder_self_attention_block = MultiHeadAttentionBlock(d_model, h, dropout)
    feed_forward_block = FeedForwardBlock(d_model, d_ff, dropout)
    decoder_block = DecoderBlock(decoder_self_attention_block, feed_forward_block, dropout)
    decoder_blocks.append(decoder_block)

  decoder = Decoder(nn.ModuleList(decoder_blocks))

  projection_layer = ProjectionLayer(d_model, vocab_size)

  transformer = Transformer(decoder, embed, pos, projection_layer)

  for p in transformer.parameters():
    if p.dim() > 1:
      nn.init.xavier_uniform_(p)

  return transformer

# **Dataset.py**

In [2]:
from typing import Any
import torch
import torch.nn as nn
from torch.utils.data import Dataset

def causal_mask(size):
  mask = torch.triu(torch.ones(1, size, size), diagonal = 1).type(torch.int)
  return mask == 0

class BilingualDataset(Dataset):
  def __init__(self, ds, seq_len)->None:
    super().__init__()

    self.ds = ds
    self.seq_len = seq_len


  def __len__(self):
    return len(self.ds)



  def __getitem__(self, index):
    item = self.ds[index]
    input_ids = item["input_ids"]
    labels = item["labels"]
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.int64),
        "labels": torch.tensor(labels, dtype=torch.int64),
        "mask": causal_mask(len(input_ids)).to(torch.int64),
    }
    

# **Config.py**

In [3]:
from pathlib import Path
def get_config():
  return {
      "batch_size": 16,
      "num_epochs": 3,
      "lr": 10**-4,
      "seq_len": 350,
      "d_model": 512,
      "model_folder": "weights",
      "model_basename": "tmodel_",
      "preload": True,
      "tokenizer_file": "tokenizer_.json",
      "experiment_name": "runs/tmodel"
  }

def get_weights_file_path(config, epoch: str):
  model_folder = Path(config['model_folder'])
  model_filename = Path(config["model_basename"])
  model_folder.mkdir(exist_ok = True)
  return str(Path(".") / model_folder / model_filename)

# **Train.py**

In [4]:
from datasets.table import list_table_cache_files
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

#from dataset import BilingualDataset, causal_mask
#from model import build_transformer
#from config import get_config, get_weights_file_path


from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.trainers import WordLevelTrainer
from tokenizers.pre_tokenizers import Whitespace

from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
import warnings

from pathlib import Path

def get_all_sentences(ds):
  for item in ds:
    yield item['text']


def get_or_build_tokenizer(config, ds):
  tokenizer_path = Path(config['tokenizer_file'])
  if not Path.exists(tokenizer_path):
    tokenizer = Tokenizer(WordLevel(unk_token = '[UNK]'))
    tokenizer.pre_tokenizer = Whitespace()
    trainer = WordLevelTrainer(special_tokens = ['[UNK]', '[EOS]'], min_frequency = 2)
    tokenizer.train_from_iterator(get_all_sentences(ds), trainer = trainer)
    tokenizer.save(str(tokenizer_path))
  else:
    tokenizer = Tokenizer.from_file(str(tokenizer_path))
  return tokenizer


def greedy_decode(model, decoder_input, tokenizer, max_len, device):
    eos_idx = tokenizer.token_to_id('[EOS]')

    while True:
        if decoder_input.size(1) +25 == max_len:
            break

        # build mask for target
        decoder_mask = causal_mask(decoder_input.size(1)).type_as(decoder_input).to(device)

        # calculate output
        out = model.decode(decoder_input, decoder_mask)

        # get next token
        prob = model.project(out[:, -1])
        _, next_word = torch.max(prob, dim=1)
        decoder_input = torch.cat(
            [decoder_input, torch.empty(1, 1).type_as(decoder_input).fill_(next_word.item()).to(device)], dim=1
        )

        if next_word == eos_idx:
            break

    return decoder_input.squeeze(0)


def run_validation(model, validation_ds, tokenizer, max_len, device, print_msg, global_step, writer, num_examples=8):
    model.eval()
    count = 0

    try:
        # get the console window width
        with os.popen('stty size', 'r') as console:
            _, console_width = console.read().split()
            console_width = int(console_width)
    except:
        # If we can't get the console width, use 80 as default
        console_width = 80

    with torch.no_grad():
        for batch in validation_ds:
            count += 1

            decoder_input = batch["input_ids"]
            if decoder_input.size(-1)<25:
              continue

            decoder_input = decoder_input[:, :torch.randint(12, 24, size=(1,)).item()].to(device) # (b, seq_len)
            prompt = tokenizer.decode(decoder_input.squeeze().detach().cpu().numpy())

            # check that the batch size is 1
            assert decoder_input.size(
                0) == 1, "Batch size must be 1 for validation"

            model_out = greedy_decode(model, decoder_input, tokenizer, max_len, device)

            labels = batch["labels"][0]
            labels = tokenizer.decode(labels.squeeze().numpy())
            model_out_text = tokenizer.decode(model_out.detach().cpu().numpy())

            # Print the source, target and model output
            print_msg('-'*console_width)
            print_msg(f"{f'PROMPT: ':>12}{prompt}")
            print_msg(f"{f'PREDICTED: ':>12}{model_out_text}")
            print_msg(f"{f'ACTUAL: ':>12}{labels}")


            if count == num_examples:
                print_msg('-'*console_width)
                break


def causal_mask(size):
    mask = torch.triu(torch.ones(1, size, size), diagonal=1).type(torch.int)
    return mask == 0


def group_texts(examples, seq_len=350):
    ids = sum(examples["input_ids"], [])
    total_length = ((len(ids) - 1) // seq_len) * seq_len

    return {
        "input_ids": [ids[i : i + seq_len] for i in range(0, total_length, seq_len)],
        "labels":    [ids[i + 1 : i + seq_len + 1] for i in range(0, total_length, seq_len)],
    }

def get_ds(config):
    ds_raw = load_dataset("roneneldan/TinyStories", split="train+validation")

    tokenizer = get_or_build_tokenizer(config, ds_raw)
    seq_len = config["seq_len"]

    # 1. Map the tokenization and grouping on the FULL dataset first
    def tokenize_function(examples):
        return {"input_ids": [tokenizer.encode(text).ids + [tokenizer.token_to_id("[EOS]")] for text in examples["text"]]}

    tokenized_ds = ds_raw.map(
        tokenize_function, batched=True, remove_columns=ds_raw.column_names
    )

    chunked_ds = tokenized_ds.map(lambda ex: group_texts(ex, seq_len), batched=True)

    # 2. Now split the preprocessed Hugging Face dataset into train and validation
    train_ds_size = int(0.2 * len(chunked_ds))
    val_ds_size = len(chunked_ds) - train_ds_size
    train_subset, val_subset = random_split(chunked_ds, [train_ds_size, val_ds_size])

    # 3. Wrap them in your custom BilingualDataset
    train_dataset = BilingualDataset(train_subset, seq_len)
    val_dataset = BilingualDataset(val_subset, seq_len)

    print(f"[Info] Total number of train samples: {len(train_dataset)}")
    print(f"[Info] Total number of validation samples: {len(val_dataset)}")

    train_dataloader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True)
    val_dataloader = DataLoader(val_dataset, batch_size=1, shuffle=True)

    return train_dataloader, val_dataloader, tokenizer

def get_model(config, vocab_len ):
  model = build_transformer(vocab_len, config['seq_len'], d_model = config['d_model'])
  return model



def train_model(config):
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  print(f"[Info] Using device: {device}")

  Path(config['model_folder']).mkdir(parents = True, exist_ok = True)

  train_dataloader, val_dataloader, tokenizer = get_ds(config)
  model = get_model(config, tokenizer.get_vocab_size(), ).to(device)

  # Tensorboard
  writer = SummaryWriter(config['experiment_name'])

  optimizer = torch.optim.Adam(model.parameters(), lr = config['lr'], eps = 1e-9)

  initial_epoch = 0
  global_step = 0
  if config['preload']:

    model_filename = get_weights_file_path(config, config['preload'])
    if Path.exists(Path(model_filename)):
      print(f"[Info] Preloading model {model_filename}")
      state = torch.load(model_filename)
      model.load_state_dict(state['model_state_dict'])
      initial_epoch = state['epoch'] + 1
      optimizer.load_state_dict(state['optimizer_state_dict'])
      global_step = state['global_step']

  loss_fn = nn.CrossEntropyLoss(label_smoothing = 0.1).to(device)

  for epoch in range(initial_epoch, config['num_epochs']):

    batch_iterator = tqdm(train_dataloader, desc = f"processing Epoch {epoch + 1 :02d}")
    for batch in batch_iterator:
      model.train()
      optimizer.zero_grad()
      decoder_input = batch['input_ids'].to(device)
      decoder_mask = batch['mask'].to(device)
      decoder_labels = batch['labels'].to(device)


      decoder_output = model.decode(decoder_input, decoder_mask)
      proj_output = model.project(decoder_output)

      #label = batch['label'].to(device)
      loss = loss_fn(proj_output.view(-1, tokenizer.get_vocab_size()), decoder_labels.view(-1))
      batch_iterator.set_postfix({"loss": f"{loss.item():6.3f}"})

      writer.add_scalar('train_loss', loss.item(), global_step)
      writer.flush()

      loss.backward()
      optimizer.step()
      global_step += 1

    run_validation(model, val_dataloader, tokenizer, config['seq_len'], device, lambda msg: batch_iterator.write(msg), global_step, writer)
    model_filename = get_weights_file_path(config, f"{epoch:02d}")
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'global_step': global_step
    }, model_filename)



In [5]:
if __name__ == "__main__":
  warnings.filterwarnings("ignore")
  config = get_config()
  train_model(config)

[Info] Using device: cuda


README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…): reconstructing file:   0%|          |  0.00B /  249MB            

data/train-00000-of-00004-2d5a1467fff108(…): downloading bytes:           |  0.00B            

data/train-00001-of-00004-5852b56a2bd28f(…): reconstructing file:   0%|          |  0.00B /  248MB            

data/train-00001-of-00004-5852b56a2bd28f(…): downloading bytes:           |  0.00B            

... (13 outputs left out)

processing Epoch 02: 100%|██████████| 16141/16141 [3:32:57<00:00,  1.26it/s, loss=3.008]


--------------------------------------------------------------------------------
    PROMPT: and started to search together again . This time , they found a big hill to roll down . They had
 PREDICTED: and started to search together again . This time , they found a big hill to roll down . They had so much fun rolling down the hill together . When it was time to go home , they said goodbye to the big hill and promised to come back again soon .
    ACTUAL: started to search together again . This time , they found a big hill to roll down . They had so much fun ! In the end , Tim and Ben learned that they should not feel ashamed and should always work together . They became best friends and had many more fun adventures in the woods . Once upon a time , there was a boy named Tim . Tim liked to play outside in the rain . One day , while it was wet , he saw a big tree . He wanted to make a toy with the wood from the tree . Tim asked his dad , " Can we cut the tree , please ?" His dad got an a

processing Epoch 03: 100%|██████████| 16141/16141 [3:31:24<00:00,  1.27it/s, loss=2.907]


--------------------------------------------------------------------------------
    PROMPT: said to her mom , " The mandala was so much fun ! Can we go
 PREDICTED: said to her mom , " The mandala was so much fun ! Can we go to the park now ?" Her mom smiled and said , " Yes , we can go to the park now . But first , let ' s go to the park ." Lily was so excited to go to the park . She ran to the swings and started to swing back and forth . She felt so free and happy . When they got to the park , Lily saw a big slide . She ran to it and climbed up the ladder . She slid down the slide and laughed . She felt so free and happy . She said to her mom , " Mom , I love the slide . It ' s so much fun !" Her mom smiled and said , " I ' m glad you enjoyed it . Let ' s go home now ."
    ACTUAL: to her mom , " The mandala was so much fun ! Can we go back to the park tomorrow to make another one ?" Her mom said with a smile , " Of course we can !" The next day , Sophia and her mom returned to the p